# Fillweight Analysis Demo: Comprehensive Lane × Phase Example

This notebook demonstrates a complete ProcessBehavior analysis of fillweight data stratified by lane and phase.

**Analysis Structure:**
- **Grouping**: 4 lanes × 2 phases = 8 combinations
- **Time variable**: Pull number (1-100)
- **Response**: Fill weight

**We'll create three complementary views:**
1. **Xbar Chart**: Compare mean fillweight across the 8 lane×phase combinations
2. **S Chart**: Compare variation across the 8 lane×phase combinations
3. **Stratified IMR Charts**: Time series tracking for each of the 8 combinations independently

In [1]:


from processbehavior import ProcessBehavior

# Load fillweight data
pb = ProcessBehavior.read_csv('../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')
pb.data.head(10)



,pull,lane,phase,fill_weight
0,1,1,1,236.93
1,1,1,2,237.39
2,1,2,1,236.30
3,1,2,2,241.35
4,1,3,1,236.09
5,1,3,2,232.30
6,1,4,1,235.81
7,1,4,2,241.89
8,2,1,1,239.67
9,2,1,2,236.39


## Part 1: Xbar/S Analysis (Comparing Lane × Phase Combinations)

First, we'll use Xbar and S charts to compare the 8 lane×phase combinations:
- **Xbar**: Are the mean fillweights different across combinations?
- **S**: Is the variation different across combinations?

In [2]:
# Create ProcessBehavior


# Step 1: formulate() - analyze data structure
study = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.lane, pb.cols.phase],
    time=pb.cols.pull
)

print(f"SDS Detected: {study.sds}")
print(f"Recommended chart: {study.recommended_chart}")
print(f"Valid charts: {study.valid_charts}")

# Step 2: execute() - run Xbar/S charts
result_xbar = study.execute(chart='Xbar')
result_xbar.charts
print(f"\nAnalysis Type: {result_xbar.summary['analysis_type']}")
print(f"Charts Created: {result_xbar.all_charts}")
print(f"Total Observations: {len(result_xbar.dataset)}")

SDS Detected: 2
Recommended chart: Xbar
Valid charts: ['Xbar', 'S', 'Imr', 'R']

Analysis Type: Xbar
Charts Created: ['Xbar', 'S']
Total Observations: 789


### Xbar/S Statistics Summary

In [3]:
# Extract Xbar and S statistics using the new API
xbar_data = result_xbar.get_chart('Xbar')[['rsg', 'xbar', 'lpl', 'upl']].copy()
sbar_data = result_xbar.get_chart('S')[['rsg', 's']].copy()

# Combine into summary table
summary_stats = xbar_data.merge(sbar_data, on='rsg')

# Add observation counts
counts = result_xbar.dataset.groupby('rsg', observed=True).size().reset_index(name='n')
summary_stats = summary_stats.merge(counts, on='rsg')

# Sort by natural order
summary_stats = summary_stats.sort_values('rsg')

# Rename columns for display
summary_stats = summary_stats.rename(columns={'lpl': 'LPL', 'upl': 'UPL'})

print("Lane x Phase Xbar/S Statistics:")
print("=" * 70)
print(summary_stats.to_string(index=False))
print("=" * 70)
print(f"Total: {summary_stats['n'].sum()} observations across {len(summary_stats)} combinations")

# Check for signals
signals = result_xbar.detect_signals(chart='Xbar')
print(f"\nXbar Signals Detected: {signals.count}")
signals_s = result_xbar.detect_signals(chart='S')
print(f"Sbar Signals Detected: {signals_s.count}")

Lane x Phase Xbar/S Statistics:
rsg    xbar     LPL     UPL     s   n
1_1 238.111 237.393 238.172 1.241  99
1_2 238.688 237.391 238.174 0.977  98
2_1 237.395 237.395 238.170 0.902 100
2_2 238.944 237.391 238.174 1.043  98
3_1 236.498 237.393 238.172 1.619  99
3_2 236.959 237.391 238.174 1.637  98
4_1 237.587 237.393 238.172 1.442  99
4_2 238.096 237.391 238.174 1.449  98
Total: 789 observations across 8 combinations

Xbar Signals Detected: 4
Sbar Signals Detected: 4


### Xbar Chart: Compare Mean Fillweights

In [8]:
# Plot Xbar chart
result_xbar.plot(chart='Xbar').show()

### S Chart: Compare Variation

In [9]:
# Plot S chart
result_xbar.plot(chart='S').show()

## Part 2: Stratified IMR Analysis (Time Series per Combination)

Now we'll create separate IMR charts for each lane×phase combination:
- Each combination gets its own control chart
- Each has independent control limits appropriate for that specific stream
- Perfect for ongoing monitoring of each combination over time

In [5]:
# Run IMR analysis using the same study - creates separate charts per combination
result_imr = study.execute(chart='Imr')

print(f"Analysis Type: {result_imr.summary['analysis_type']}")
print(f"Number of IMR Charts Created: {len(result_imr.all_charts)}")
print(f"Chart Keys (naturally sorted): {result_imr.all_charts}")
print(f"\nEach chart represents one lane×phase combination tracked over time")
print(f'all charts: {result_imr.all_charts}')

Analysis Type: Imr
Number of IMR Charts Created: 2
Chart Keys (naturally sorted): ['Imr', 'R']

Each chart represents one lane×phase combination tracked over time
all charts: ['Imr', 'R']


### Stratified IMR Statistics Summary

### Stratified IMR Charts: All 8 Combinations

Faceted view showing all 8 IMR charts simultaneously:

In [7]:
# Plot all 8 IMR charts in a faceted grid
result_imr.plot(facet=True, ncols=4).show()

### Example: Drill Down into a Specific Combination

You can easily access and plot any specific combination:

In [12]:
# Example: Plot just Lane 1, Phase 1
print("Drilling down into Lane 1, Phase 1:")
print(f"Available chart keys: {result_imr.all_charts}")
print(f"Available strata keys: {result_imr.strata}")
print(f"\nAccessing chart: '1_1'")
result_1_1 = result_imr.focus('1_1')
result_1_1.plot().show()
# Get data for Lane 1, Phase 1 using new API
#lane1_phase1_data = result_imr.get_chart('1_1')
#lane1_phase1_stats = result_imr.get_statistics('1_1')
#print(f"Observations: {len(lane1_phase1_data)}")
#print(f"Center: {lane1_phase1_stats['center']:.3f}")
#print(f"Control Limits: [{lane1_phase1_stats['lpl']:.3f}, {lane1_phase1_stats['upl']:.3f}]")

# Plot single chart
#result_imr.plot(chart='1_1').show()

Drilling down into Lane 1, Phase 1:
Available chart keys: ['Imr', 'R']
Available strata keys: ['1_1', '1_2', '2_1', '2_2', '3_1', '3_2', '4_1', '4_2']

Accessing chart: '1_1'


## Summary: Comprehensive Analysis Results

### What We Accomplished

**1. Xbar/S Analysis (Group Comparison):**
- Compared mean fillweight across 8 lane×phase combinations
- Identified which combinations have higher/lower means
- Compared variation (S chart) across combinations
- Used common control limits to identify out-of-spec combinations

**2. Stratified IMR Analysis (Time Series Monitoring):**
- Created 8 separate IMR charts (one per combination)
- Each combination has its own appropriate control limits
- Can track each combination's behavior over time
- All charts pre-calculated and instantly accessible

### Key Insights

**Complementary Perspectives:**
- **Xbar/S answers:** "Which lane×phase combinations differ from each other?"
- **Stratified IMR answers:** "How does each combination behave over time?"

**ProcessBehavior's Power:**
- Single `formulate()` call analyzes structure
- Multiple chart types from same Study
- Natural sorting ensures logical ordering ('1_1', '1_2', '2_1', not '1_1', '1_2', '10_1')
- All 8 IMR charts pre-calculated and ready to access
- Beautiful faceted plots with one line of code

### Code Simplicity

```python
# Step 1: formulate() - analyze structure once
study = pdf.formulate(
    response=pdf.columns.fill_weight,
    factors=[pdf.columns.lane, pdf.columns.phase],
    time=pdf.columns.pull
)

# Step 2: analyze() - run different chart types
result_xbar = study.analyze(chart='Xbar')  # Compare combinations
result_imr = study.analyze(chart='Imr')    # Track each combination

# Plot it
result_imr.plot(facet=True, ncols=4)
```

**That's the complete power of ProcessBehavior!**